# Template notebook

You’re working with **Northstar Desk**, a UK-based subscription software company. Over the last year, the team has handled a steady stream of operational cases: billing and renewals, access and admin requests, reporting issues, integrations, bugs, and occasional performance incidents. The dataset you’ve been given is a snapshot export from Northstar’s case management system: each row is one case, with structured triage fields, outcomes, timings, and a short free-text summary.

Northstar doesn’t want a “fully automated AI that replaces support staff”. They want something more practical: a tool that helps humans **make better decisions, faster**, using the data they already have.

## Your mission

Build a **decision-support prototype** that improves how a frontline or operational user handles cases. Your tool can focus on any part of the workflow, as long as it produces something a real person could use.

Your prototype should support at least one of these outcomes:

- **Understand** what’s coming in (themes, clusters, common issues, emerging patterns)  
- **Prioritise** what to look at first (risk signals, urgency proxies, workload triage)  
- **Route** work more effectively (suggest team/category/subcategory, detect likely escalations)  
- **Resolve** issues faster (surface likely next steps, similar cases, common fixes)  
- **Learn** from outcomes (what tends to lead to long resolution times, escalations, poor CSAT)

You can interpret “decision-support” broadly. The key is that it helps a user do something concrete with the case data.

## Prototype requirement (this is the steer)

Your prototype must be an **interactive tool**, not just analysis:

- A **Gradio app** (recommended), or other small deployable interface.  
- It can be simple: one or two screens, a small number of inputs, and clear outputs.  
- It should demonstrate a realistic workflow: a user provides inputs (e.g. a case summary, category, priority, or a date range) and your tool returns helpful outputs (e.g. routing suggestions, similar cases, risk flags, or a summary of patterns).

A notebook is fine for development, but the end result should include an interface that could plausibly be shared with a non-technical colleague.

## Choose a track (optional)

If you’re having trouble choosing what to build, you can try one of the following tracks:

- **Track 1: Triage assistant** — classify, route, prioritise, or flag risk for new cases.  
- **Track 2: Ops insight tool** — explore trends and spikes, with filters a team lead would use.  
- **Track 3: Similarity + retrieval tool** — find related cases and surface “what worked before”.  
- **Track 4: Process quality tool** — analyse what drives delays/escalations and where process breaks down (optional fairness checks as audit, not decisioning).

Any track is valid. You’re judged on usefulness and clarity, not on picking the “right” one.

## What you’ll present (end of day 2)

1. **Working prototype**  
    Gradio app (or similar) demonstrating the decision-support workflow.  
2. **Short technical summary**  
    Data prep, modelling choices, and why they make sense. Risks, limitations, and interpretability choices.  
3. **Short user-facing story**  
    Who uses the tool, what problem it solves for them, and how they’d use it day-to-day.  
4. **Roadmap**  
    What you’d do next with another week, or if you were building this internally (data, modelling, UX, governance, monitoring).


## Libraries
As always, we'll start by importing the necessary libraries.

In [ ]:
# It's good practice to add comments to explain your code 
import numpy as np
import pandas as pd

In [ ]:
from pathlib import Path
import csv

DATA_DIR = Path("data")

EXPECTED_COLUMNS = [
    "case_id",
    "snapshot_at",
    "created_at",
    "channel",
    "case_type",
    "category",
    "subcategory",
    "priority",
    "sla_target_hours",
    "first_response_time_hours",
    "resolution_time_hours",
    "status",
    "resolution_code",
    "escalated",
    "assigned_team",
    "escalation_team",
    "customer_tenure_months",
    "plan_tier",
    "region_uk",
    "age_band",
    "gender",
    "case_summary",
    "sentiment",
    "csat_score",
    "tags",
]


def repair_row(row, source_file, row_number):
    repair = None

    if len(row) == len(EXPECTED_COLUMNS) - 1:
        #Assumed based on initial analysis that this is missing gender - if more time would add additional checks - such as checking age band looks right, or gender in ['M', 'F']
        row = row[:20] + [""] + row[20:]
        repair = {
            "source_file": source_file,
            "row_number": row_number,
            "issue": "missing_gender_column",
            "action": "inserted_blank_gender_before_case_summary",
        }

    if len(row) != len(EXPECTED_COLUMNS):
        raise ValueError(
            f"{source_file} row {row_number} has {len(row)} columns after repair; "
            f"expected {len(EXPECTED_COLUMNS)}"
        )

    return row, repair


def load_case_data(data_dir=DATA_DIR):
    rows = []
    repairs = []

    for path in sorted(data_dir.glob("*.csv")):
        with path.open(newline="", encoding="utf-8-sig") as file:
            reader = csv.reader(file)
            header = next(reader)

            if header != EXPECTED_COLUMNS:
                raise ValueError(f"Unexpected header in {path.name}: {header}")

            for row_number, row in enumerate(reader, start=2):
                row, repair = repair_row(row, path.name, row_number)
                rows.append(dict(zip(EXPECTED_COLUMNS, row)) | {"source_file": path.name})
                if repair is not None:
                    repairs.append(repair)

    df = pd.DataFrame(rows)

    for column in df.columns:
        if df[column].dtype == "object":
            df[column] = df[column].str.strip()
    df = df.replace("", pd.NA)

    for column in ["snapshot_at", "created_at"]:
        df[column] = pd.to_datetime(df[column], errors="coerce", utc=True)

    numeric_columns = [
        "sla_target_hours",
        "first_response_time_hours",
        "resolution_time_hours",
        "customer_tenure_months",
        "csat_score",
    ]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    df["escalated"] = df["escalated"].str.lower().map({"true": True, "false": False})
    df["tags"] = df["tags"].apply(
        lambda value: [tag.strip() for tag in str(value).split(";") if tag.strip()]
        if pd.notna(value)
        else []
    )

    repair_log = pd.DataFrame(repairs)
    return df, repair_log


cases, repair_log = load_case_data()

print(f"Loaded {cases.shape[0]:,} cases from {cases['source_file'].nunique()} files")
print(f"Applied {len(repair_log):,} row repairs")
display(repair_log.value_counts(["source_file", "issue"]).rename("repairs").reset_index())
display(cases.head())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

ROUTING_FEATURES = ["channel", "priority", "sla_target_hours", "plan_tier", "sentiment", "tags"]
ETHICS_ANALYSIS_COLUMNS = ["customer_tenure_months", "region_uk", "age_band", "gender"]
FUTURE_TARGETS = ["case_type", "category", "subcategory"]
FUTURE_TEXT_FEATURES = ["case_summary"]

MONTH_ORDER = {
    "Jan": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12,
}
TRAIN_MONTHS = range(1, 7)
TEST_MONTHS = range(7, 10)
HOLDOUT_MONTHS = range(10, 13)
PRIMARY_EVALUATION_SLICE = "solved_only"


def encode_routing_features(df, reference_columns=None, sla_fill_value=None):
    features = df[ROUTING_FEATURES].copy()

    tag_values = features.pop("tags").apply(lambda value: value if isinstance(value, list) else [])
    tag_dummies = pd.get_dummies(tag_values.explode()).groupby(level=0).max()
    tag_dummies = tag_dummies.reindex(features.index, fill_value=0).add_prefix("tag__")

    categorical_features = pd.get_dummies(
        features[["channel", "priority", "plan_tier", "sentiment"]],
        dummy_na=True,
    )
    if sla_fill_value is None:
        sla_fill_value = features["sla_target_hours"].median()
    numeric_features = features[["sla_target_hours"]].fillna(sla_fill_value)

    encoded = pd.concat([categorical_features, numeric_features, tag_dummies], axis=1).fillna(0)

    if reference_columns is not None:
        encoded = encoded.reindex(columns=reference_columns, fill_value=0)

    return encoded


def add_file_month(df):
    df = df.copy()
    df["file_month_name"] = df["source_file"].str.extract(r"Q\d-([A-Za-z]+)\.csv")[0]
    df["file_month"] = df["file_month_name"].map(MONTH_ORDER)

    if df["file_month"].isna().any():
        unknown_files = df.loc[df["file_month"].isna(), "source_file"].unique()
        raise ValueError(f"Could not infer month from source_file values: {unknown_files}")

    return df


def build_temporal_model_frames(labelled_cases, target_column):
    labelled_cases = add_file_month(labelled_cases)

    split_masks = {
        "train_jan_to_jun": labelled_cases["file_month"].isin(TRAIN_MONTHS),
        "test_july_to_sept": labelled_cases["file_month"].isin(TEST_MONTHS),
        "holdout_oct_to_dec": labelled_cases["file_month"].isin(HOLDOUT_MONTHS),
    }

    split_cases = {
        split_name: labelled_cases.loc[mask].copy()
        for split_name, mask in split_masks.items()
    }

    train_cases = split_cases["train_jan_to_jun"]
    sla_fill_value = train_cases["sla_target_hours"].median()
    X_train = encode_routing_features(train_cases, sla_fill_value=sla_fill_value)
    feature_columns = X_train.columns

    frames = {}
    for split_name, cases_for_split in split_cases.items():
        if split_name == "train_jan_to_jun":
            X = X_train
        else:
            X = encode_routing_features(
                cases_for_split,
                reference_columns=feature_columns,
                sla_fill_value=sla_fill_value,
            )

        frames[split_name] = {
            "cases": cases_for_split,
            "X": X,
            "y": cases_for_split[target_column].copy(),
        }

    return frames


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
def make_model(model_type):
    if model_type == "logistic":
        return LogisticRegression(max_iter=1000, class_weight="balanced")

    if model_type == "decision_tree":
        return DecisionTreeClassifier(
            random_state=42,
            class_weight="balanced",
            max_depth=8,
        )

    if model_type == "random_forest":
        return RandomForestClassifier(
            random_state=42,
            class_weight="balanced",
            n_estimators=300,
            max_depth=None,
        )

    raise ValueError(
        "model_type must be one of: 'logistic', 'decision_tree', 'random_forest'"
    )


In [ ]:
def train_temporal_classifier(frames, target_name, model_type="logistic", slice_name=None):
    X_train = frames["train_jan_to_jun"]["X"]
    y_train = frames["train_jan_to_jun"]["y"]
    X_test = frames["test_july_to_sept"]["X"]
    y_test = frames["test_july_to_sept"]["y"]
    X_holdout = frames["holdout_oct_to_dec"]["X"]

    if y_train.nunique() < 2:
        raise ValueError(f"{target_name} needs at least two training classes")

    model = make_model(model_type)
    model.fit(X_train, y_train)

    label = f"{target_name} {model_type}"
    if slice_name is not None:
        label = f"{label} ({slice_name})"

    print(f"\n{label}: train Jan-Jun, test July-Sept")
    print(
        f"Training rows: {len(X_train):,}; "
        f"Q3 test rows: {len(X_test):,}; "
        f"Q4 holdout rows: {len(X_holdout):,}; "
        f"features: {X_train.shape[1]:,}"
    )

    test_accuracy = None

    if X_test.empty:
        print("No labelled Q3 rows are available for this target, so July-Sept cannot be used as a test set.")
    else:
        test_predictions = model.predict(X_test)
        test_accuracy = (test_predictions == y_test).mean()
        print(classification_report(y_test, test_predictions, zero_division=0))

    metrics = {
        "target": target_name,
        "slice": slice_name,
        "model_type": model_type,
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "holdout_rows": len(X_holdout),
        "test_accuracy": test_accuracy,
    }

    return model, metrics


assigned_team_case_slices = {
    "solved_only": cases["status"].eq("solved"),
    "active_only": cases["status"].ne("solved"),
    "all_cases": cases["status"].notna(),
}

assigned_team_frames_by_slice = {}
for slice_name, slice_mask in assigned_team_case_slices.items():
    slice_cases = cases[
        slice_mask & cases["assigned_team"].notna()
    ].copy()
    assigned_team_frames_by_slice[slice_name] = build_temporal_model_frames(
        slice_cases,
        "assigned_team",
    )

escalation_team_training_cases = cases[
    cases["escalated"].eq(True) & cases["escalation_team"].notna()
].copy()
escalation_team_frames = build_temporal_model_frames(
    escalation_team_training_cases,
    "escalation_team",
)


In [ ]:
models = {}
evaluation_rows = []

for slice_name, frames in assigned_team_frames_by_slice.items():
    print("=" * 80)
    print(f"Assigned-team evaluation slice: {slice_name}")

    for model_type in ["logistic", "decision_tree", "random_forest"]:
        model, metrics = train_temporal_classifier(
            frames,
            "assigned_team",
            model_type=model_type,
            slice_name=slice_name,
        )
        models[("assigned_team", slice_name, model_type)] = model
        evaluation_rows.append(metrics)

assigned_team_model = models[("assigned_team", PRIMARY_EVALUATION_SLICE, "decision_tree")]

# Escalation-team labels only exist for escalated cases, so keep this as a separate routing task.
escalation_team_model, escalation_metrics = train_temporal_classifier(
    escalation_team_frames,
    "escalation_team",
    model_type="logistic",
    slice_name="escalated_with_known_team",
)
models[("escalation_team", "escalated_with_known_team", "logistic")] = escalation_team_model
evaluation_rows.append(escalation_metrics)

ethics_analysis_frame = cases[
    ["case_id", "status", "escalated", "assigned_team", "escalation_team"]
    + ETHICS_ANALYSIS_COLUMNS
].copy()

evaluation_summary = pd.DataFrame(evaluation_rows)
print("\nModel evaluation summary")
display(evaluation_summary)

split_summary_rows = []
for slice_name, frames in assigned_team_frames_by_slice.items():
    for split_name, split in frames.items():
        split_summary_rows.append(
            {
                "target": "assigned_team",
                "slice": slice_name,
                "split": split_name,
                "rows": len(split["y"]),
                "classes": split["y"].nunique(),
                "files": ", ".join(sorted(split["cases"]["source_file"].unique())),
            }
        )

for split_name, split in escalation_team_frames.items():
    split_summary_rows.append(
        {
            "target": "escalation_team",
            "slice": "escalated_with_known_team",
            "split": split_name,
            "rows": len(split["y"]),
            "classes": split["y"].nunique(),
            "files": ", ".join(sorted(split["cases"]["source_file"].unique())),
        }
    )

print("\nTemporal split summary")
display(pd.DataFrame(split_summary_rows))

print("Primary assigned-team evaluation slice:", PRIMARY_EVALUATION_SLICE)
print("Ethics-only columns kept out of model features:", ETHICS_ANALYSIS_COLUMNS)
print("Future modelling candidates:", FUTURE_TARGETS + FUTURE_TEXT_FEATURES)


In [ ]:
cases, repair_log = load_case_data()

In [ ]:
cases